# Retail Sales Data Cleaning

## Project Objective

The purpose of this notebook is to clean and prepare the retail sales dataset for exploratory data analysis, SQL analysis, and dashboard development.

The cleaning process includes:

- inspecting the dataset
- correcting data types
- checking missing values
- removing duplicate records
- validating numerical values
- creating useful analytical fields
- exporting a cleaned dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [9]:
data_path = Path("../data/raw/Superstore_Sales.csv")

df = pd.read_csv(data_path, encoding="latin1")

## Initial Data Inspection

The first step is to inspect several rows of the dataset to understand its structure, column names, and values.

In [10]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 9,994
Number of columns: 21


In [11]:
df.columns.tolist()

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [12]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   row_id         9994 non-null   int64  
 1   order_id       9994 non-null   object 
 2   order_date     9994 non-null   object 
 3   ship_date      9994 non-null   object 
 4   ship_mode      9994 non-null   object 
 5   customer_id    9994 non-null   object 
 6   customer_name  9994 non-null   object 
 7   segment        9994 non-null   object 
 8   country        9994 non-null   object 
 9   city           9994 non-null   object 
 10  state          9994 non-null   object 
 11  postal_code    9994 non-null   int64  
 12  region         9994 non-null   object 
 13  product_id     9994 non-null   object 
 14  category       9994 non-null   object 
 15  sub_category   9994 non-null   object 
 16  product_name   9994 non-null   object 
 17  sales          9994 non-null   float64
 18  quantity

In [14]:
#Check missing values
missing_values = df.isna().sum().sort_values(ascending=False)

missing_values

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

### Missing Value Assessment

The dataset was checked for missing values. Columns with missing observations were reviewed to determine whether rows should be removed, values should be imputed, or the missing values should remain unchanged.

In [15]:
#Check duplicate rows
duplicate_count = df.duplicated().sum()

print(f"Number of fully duplicated rows: {duplicate_count}")

Number of fully duplicated rows: 0


In [16]:
#Check whether row_id is unique
print(f"Unique Row IDs: {df['row_id'].nunique():,}")
print(f"Total Rows: {len(df):,}")

Unique Row IDs: 9,994
Total Rows: 9,994


In [18]:
#inspect the date format:
df[["order_date", "ship_date"]].head(10)

,order_date,ship_date
0,11/8/2016,11/11/2016
1,11/8/2016,11/11/2016
2,6/12/2016,6/16/2016
3,10/11/2015,10/18/2015
4,10/11/2015,10/18/2015
5,6/9/2014,6/14/2014
6,6/9/2014,6/14/2014
7,6/9/2014,6/14/2014
8,6/9/2014,6/14/2014
9,6/9/2014,6/14/2014


In [20]:
#Convert them:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    errors="coerce"
)

df[["order_date", "ship_date"]].dtypes

order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object

In [22]:
#Check whether conversion produced missing dates:
print("Invalid order dates:", df["order_date"].isna().sum())
print("Invalid ship dates:", df["ship_date"].isna().sum())

Invalid order dates: 0
Invalid ship dates: 0


In [25]:
#Validate shipping dates
#A shipment should generally not happen before the order date.
#The shipping duration creation:
df["shipping_days"] = (
    df["ship_date"] - df["order_date"]
).dt.days

invalid_shipping = df[df["shipping_days"] < 0]

print(f"Orders shipped before order date: {len(invalid_shipping)}")

Orders shipped before order date: 0


In [26]:
#inspect long shipping periods:
df.sort_values("shipping_days", ascending=False)[
    ["order_id", "order_date", "ship_date", "shipping_days"]
].head(10)

,order_id,order_date,ship_date,shipping_days
9001,CA-2014-103807,2014-12-02,2014-12-09,7
9002,CA-2014-103807,2014-12-02,2014-12-09,7
1966,CA-2017-165841,2017-12-21,2017-12-28,7
1967,CA-2017-165841,2017-12-21,2017-12-28,7
1968,CA-2017-165841,2017-12-21,2017-12-28,7
8893,CA-2014-124723,2014-08-05,2014-08-12,7
6802,CA-2016-109827,2016-12-25,2017-01-01,7
6803,CA-2016-109827,2016-12-25,2017-01-01,7
6654,CA-2017-135937,2017-02-20,2017-02-27,7
6655,CA-2017-135937,2017-02-20,2017-02-27,7


In [27]:
#Inspect numerical columns
numeric_columns = [
    "sales",
    "quantity",
    "discount",
    "profit"
]

df[numeric_columns].describe().round(2)

,sales,quantity,discount,profit
count,9994.00,9994.00,9994.00,9994.00
mean,229.86,3.79,0.16,28.66
std,623.25,2.23,0.21,234.26
min,0.44,1.00,0.00,-6599.98
25%,17.28,2.00,0.00,1.73
50%,54.49,3.00,0.20,8.67
75%,209.94,5.00,0.20,29.36
max,22638.48,14.00,0.80,8399.98


In [28]:
#Sales should normally be greater than zero
print("Zero sales values:", (df["sales"] == 0).sum())
print("Negative sales values:", (df["sales"] < 0).sum())

Zero sales values: 0
Negative sales values: 0


In [29]:
#Quantity should normally be a positive whole number
print("Zero quantities:", (df["quantity"] == 0).sum())
print("Negative quantities:", (df["quantity"] < 0).sum())

Zero quantities: 0
Negative quantities: 0


In [30]:
#Check whether quantity contains decimals:
non_integer_quantity = df[
    df["quantity"] % 1 != 0
]

print(f"Non-integer quantity values: {len(non_integer_quantity)}")

Non-integer quantity values: 0


In [32]:
#Inspect the available discount values:
sorted(df["discount"].unique())

[np.float64(0.0),
 np.float64(0.1),
 np.float64(0.15),
 np.float64(0.2),
 np.float64(0.3),
 np.float64(0.32),
 np.float64(0.4),
 np.float64(0.45),
 np.float64(0.5),
 np.float64(0.6),
 np.float64(0.7),
 np.float64(0.8)]

In [33]:
#Create a percentage version for easier interpretation:
df["discount_percentage"] = df["discount"] * 100

In [35]:
#Profit Investigation
#Check the number of loss-making transactions:
loss_transactions = df[df["profit"] < 0]

print(f"Loss-making transactions: {len(loss_transactions):,}")
print(
    f"Percentage of loss-making transactions: "
    f"{len(loss_transactions) / len(df) * 100:.2f}%"
)

Loss-making transactions: 1,871
Percentage of loss-making transactions: 18.72%


In [36]:
#View some examples of negative transaction
loss_transactions[
    [
        "order_id",
        "product_name",
        "category",
        "sales",
        "discount",
        "profit"
    ]
].sort_values("profit").head(10)

,order_id,product_name,category,sales,discount,profit
7772,CA-2016-108196,Cubify CubeX 3D Printer Double Head Print,Technology,4499.985,0.7,-6599.9780
683,US-2017-168116,Cubify CubeX 3D Printer Triple Head Print,Technology,7999.980,0.5,-3839.9904
9774,CA-2014-169019,GBC DocuBind P400 Electric Binding System,Office Supplies,2177.584,0.8,-3701.8928
3011,CA-2017-134845,Lexmark MX611dhe Monochrome Laser Printer,Technology,2549.985,0.7,-3399.9800
4991,US-2017-122714,Ibico EPK-21 Electric Binding System,Office Supplies,1889.990,0.8,-2929.4845
3151,CA-2015-147830,Cubify CubeX 3D Printer Double Head Print,Technology,1799.994,0.7,-2639.9912
5310,CA-2017-131254,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,1525.188,0.8,-2287.7820
9639,CA-2015-116638,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Furniture,4297.644,0.4,-1862.3124
1199,CA-2016-130946,GBC DocuBind P400 Electric Binding System,Office Supplies,1088.792,0.8,-1850.9464
2697,CA-2014-145317,Cisco TelePresence System EX90 Videoconferenci...,Technology,22638.480,0.5,-1811.0784


In [37]:
#categorical values
#Inspect major categorical columns:
categorical_columns = [
    "ship_mode",
    "segment",
    "region",
    "category",
    "sub_category"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


ship_mode:
ship_mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64

segment:
segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64

region:
region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64

category:
category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64

sub_category:
sub_category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64


In [39]:
#Create useful date columns
df["order_year"] = df["order_date"].dt.year
df["order_month_number"] = df["order_date"].dt.month
df["order_month"] = df["order_date"].dt.month_name()
df["order_quarter"] = df["order_date"].dt.quarter
df["order_year_month"] = df["order_date"].dt.to_period("M").astype(str)


df[
    [
        "order_date",
        "order_year",
        "order_month_number",
        "order_month",
        "order_quarter",
        "order_year_month"
    ]
].head()

,order_date,order_year,order_month_number,order_month,order_quarter,order_year_month
0,2016-11-08,2016,11,November,4,2016-11
1,2016-11-08,2016,11,November,4,2016-11
2,2016-06-12,2016,6,June,2,2016-06
3,2015-10-11,2015,10,October,4,2015-10
4,2015-10-11,2015,10,October,4,2015-10


In [40]:
#profit margin
#Profit margin shows profit as a percentage of sales.
df["profit_margin"] = np.where(
    df["sales"] != 0,
    df["profit"] / df["sales"],
    np.nan
)

df["profit_margin_percentage"] = (
    df["profit_margin"] * 100
)

df[
    ["sales", "profit", "profit_margin_percentage"]
].head()

,sales,profit,profit_margin_percentage
0,261.9600,41.9136,16.00
1,731.9400,219.5820,30.00
2,14.6200,6.8714,47.00
3,957.5775,-383.0310,-40.00
4,22.3680,2.5164,11.25


In [43]:
#Create a profitability label
df["profit_status"] = np.select(
    [
        df["profit"] > 0,
        df["profit"] == 0,
        df["profit"] < 0
    ],
    [
        "Profitable",
        "Break-even",
        "Loss"
    ],
    default="Unknown"
)

df["profit_status"].value_counts()

profit_status
Profitable    8058
Loss          1871
Break-even      65
Name: count, dtype: int64

In [46]:
#Outliers
#We use the IQR method to identify them:
def calculate_outlier_summary(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = dataframe[
        (dataframe[column] < lower_bound) |
        (dataframe[column] > upper_bound)
    ]

    return {
        "column": column,
        "lower_bound": round(lower_bound, 2),
        "upper_bound": round(upper_bound, 2),
        "outlier_count": len(outliers)
    }

    #Run it for sales and profit:
    outlier_results = pd.DataFrame([
    calculate_outlier_summary(df, "sales"),
    calculate_outlier_summary(df, "profit")
    ])

    outlier_results

### Outlier Treatment

Potential outliers were identified using the interquartile range method. They were retained because unusually large orders, profits, and losses may represent legitimate business transactions rather than data-entry errors.

In [47]:
#final quality check
print("Final dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing order dates:", df["order_date"].isna().sum())
print("Missing ship dates:", df["ship_date"].isna().sum())
print("Negative shipping days:", (df["shipping_days"] < 0).sum())
print("Invalid discounts:", ((df["discount"] < 0) | (df["discount"] > 1)).sum())
print("Non-positive quantities:", (df["quantity"] <= 0).sum())

Final dataset shape: (9994, 31)
Duplicate rows: 0
Missing order dates: 0
Missing ship dates: 0
Negative shipping days: 0
Invalid discounts: 0
Non-positive quantities: 0


In [48]:
#Create the cleaned-data path:
output_path = Path("../data/cleaned/cleaned_retail_sales.csv")

In [49]:
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: ..\data\cleaned\cleaned_retail_sales.csv


## Data Cleaning Summary

The retail sales dataset was prepared for analysis through the following steps:

- standardized column names
- reviewed missing values
- identified and removed fully duplicated rows
- converted order and shipping dates to datetime format
- checked sales, quantity, discount, and profit values
- validated shipping durations
- retained legitimate negative-profit transactions
- reviewed potential numerical outliers
- created year, month, quarter, shipping duration, discount percentage, profit margin, and profitability fields
- exported the cleaned dataset for future analysis

The cleaned dataset is now ready for exploratory data analysis, SQL analysis, and Power BI dashboard development.